# Notebook 03 — Benchmarks and evaluation

**What you’ll do:** Define a small set of **benchmark questions** with answers checked in SQL, then add them to your **main** Genie space (the one **with** examples from **02**).

**Why it matters:** Benchmarks give you a repeatable way to see if Genie still answers correctly after you change instructions or models.

**Before you start:** **01** (data) and **02** (spaces). **Workshop order:** 01 → 02 → **03** → 04 → … → 09. **Next:** **04** (explore in the UI).


## Compute

Use **Serverless**. Catalog and schema come from **widgets** only.


## Tips for good benchmarks

- Ask for **one clear number or small result** per question (easy to score automatically).
- Compute **expected answers in SQL** from the same tables Genie uses—no manual guesswork.
- Mix **counts, averages, and filters** (for example OEE, downtime, defects).
- Use questions **real users** would ask; update benchmarks when the schema changes.
- In **06**, allow a small **tolerance** (for example 5%) for rounding.
- Sample questions in **02** teach *style*; benchmarks here track *correctness* over time.
- If the API step errors, add the same questions under Genie **Evaluation / benchmarks** in the UI.


## Configuration

Reads **`genie_space_id`** (your main configured space from **02**). Run **02** first if it is missing.


In [ ]:
dbutils.widgets.text("catalog", "workshop_demo", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
fqn = f"{CATALOG}.{SCHEMA}"

from databricks.sdk import WorkspaceClient
import html
import re
import requests

w = WorkspaceClient()
host = w.config.host.rstrip("/")
auth = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(host: str, space_id: str) -> str:
    # Genie browser UI: /genie/rooms/<space_id>?o=<workspace_id> (not /genie/spaces/...)
    h = (host or "").rstrip("/")
    sid = str(space_id or "").strip()
    m = re.search(r"adb-(\d+)\.", h)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{h}/genie/rooms/{sid}{q}"


def _cfg(key):
    rows = spark.sql(
        f"SELECT value FROM {fqn}.workshop_config WHERE key = '{key}'"
    ).collect()
    return rows[0]["value"] if rows else None


SID_PRIMARY = _cfg("genie_space_id")
if not SID_PRIMARY:
    raise RuntimeError("Missing genie_space_id in workshop_config — run notebook 02.")
print("Primary Genie space (with examples):", SID_PRIMARY)
_ui = genie_ui_room_url(host, SID_PRIMARY)
print("Primary Genie UI URL (computed):", _ui)
displayHTML(
    '<p><a href="'
    + html.escape(_ui, quote=True)
    + '" target="_blank" rel="noopener">Open primary Genie space</a></p>'
)


def push_benchmarks(benchmarks_list, space_id, label=""):
    try:
        existing = requests.get(
            f"{host}/api/2.0/data-rooms/{space_id}/curated-questions",
            headers=auth,
            params={"question_type": "BENCHMARK"},
        ).json()
        for q in existing.get("curated_questions", []):
            qid = q.get("curated_question_id") or q.get("id") or q.get("question_id")
            if qid:
                requests.delete(
                    f"{host}/api/2.0/data-rooms/{space_id}/curated-questions/{qid}",
                    headers=auth,
                )
        ok = 0
        for b in benchmarks_list:
            r = requests.post(
                f"{host}/api/2.0/data-rooms/{space_id}/curated-questions",
                headers=auth,
                json={
                    "curated_question": {
                        "data_space_id": space_id,
                        "question_text": b["q"],
                        "question_type": "BENCHMARK",
                        "answer_text": b["sql"],
                        "is_deprecated": False,
                    },
                    "data_space_id": space_id,
                },
            )
            if r.status_code == 200:
                ok += 1
        print(f"{label}: pushed {ok}/{len(benchmarks_list)} benchmarks")
    except Exception as e:
        print(f"push_benchmarks skipped: {e}")


print("Helpers loaded")


## Define benchmark suite

**v2 — harder questions:** joins, filters, ratios, and subqueries so simple table scans are not enough. The **without-examples** space often scores lower than the primary space that has curated Q→SQL and benchmarks from this notebook. Re-run **03** after changing the list so BENCHMARK rows stay in sync.


In [ ]:
# Scalar ground truth only (notebook 06 parses one number from Genie’s result).
# Use YEAR(CAST(... AS DATE)) because event_date / date are strings in the seed data.

benchmarks_v1 = [
    {
        "q": "How many distinct production lines had at least one defect_detected event in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) AS answer FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "What is the sum of downtime_minutes from quality_metrics_daily for all rows in 2024 where the plant is in Michigan (join to plants by state)?",
        "sql": f"""SELECT CAST(ROUND(SUM(q.downtime_minutes), 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(SUM(q.downtime_minutes), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
    {
        "q": "What is the total units_produced summed from quality_metrics_daily for January through March 2024 (Q1) only?",
        "sql": f"""SELECT CAST(SUM(q.units_produced) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q WHERE CAST(q.date AS DATE) >= DATE '2024-01-01' AND CAST(q.date AS DATE) < DATE '2024-04-01'""",
        "gt": f"SELECT CAST(SUM(q.units_produced) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE CAST(q.date AS DATE) >= DATE '2024-01-01' AND CAST(q.date AS DATE) < DATE '2024-04-01'",
    },
    {
        "q": "How many production lines had 500 or more unit_produced events in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t",
    },
    {
        "q": "How many safety incidents are both severity Critical and category Equipment Malfunction?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.safety_incidents WHERE severity = 'Critical' AND category = 'Equipment Malfunction'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.safety_incidents WHERE severity = 'Critical' AND category = 'Equipment Malfunction'",
    },
    {
        "q": "How many operators work at plants located in Michigan (use operators.plant_id joined to plants)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'",
    },
    {
        "q": "For calendar year 2024 quality_metrics_daily only, what is the overall defect rate in percent rounded to a whole number: 100 times the sum of defects_found divided by the sum of units_produced?",
        "sql": f"""SELECT CAST(ROUND(100.0 * SUM(defects_found) / SUM(units_produced), 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(defects_found) / SUM(units_produced), 0) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024",
    },
    {
        "q": "How many production events in 2024 have event_type inspection_passed?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.production_events WHERE event_type = 'inspection_passed' AND YEAR(CAST(event_date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'inspection_passed' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "How many quality_metrics_daily rows in 2024 have oee_score strictly below 0.70?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024 AND oee_score < 0.70""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.quality_metrics_daily WHERE YEAR(CAST(date AS DATE)) = 2024 AND oee_score < 0.70",
    },
    {
        "q": "What is the sum of employee_count for all plants in Texas?",
        "sql": f"""SELECT CAST(COALESCE(SUM(p.employee_count), 0) AS BIGINT) AS answer FROM {fqn}.plants p WHERE p.state = 'Texas'""",
        "gt": f"SELECT CAST(COALESCE(SUM(p.employee_count), 0) AS BIGINT) FROM {fqn}.plants p WHERE p.state = 'Texas'",
    },
    {
        "q": "How many equipment_feedback rows are for production lines at the EV Battery Innovation Center (join feedback to production_lines and match that plant)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'",
    },
    {
        "q": "For plant P003 only and calendar year 2024, what is average OEE as a whole percent rounded to an integer (average oee_score times 100, then round)?",
        "sql": f"""SELECT CAST(ROUND(AVG(q.oee_score) * 100, 0) AS BIGINT) AS answer FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND YEAR(CAST(q.date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(AVG(q.oee_score) * 100, 0) AS BIGINT) FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
]
print(len(benchmarks_v1), "benchmarks (v2 harder suite)")


## Verify ground truth in Spark

Run each `gt` query once. Fix definitions before pushing to Genie if any fail.


In [ ]:
for i, b in enumerate(benchmarks_v1, 1):
    try:
        v = spark.sql(b["gt"]).collect()[0][0]
        print(f"OK Q{i}: {v}")
    except Exception as e:
        print(f"FAIL Q{i}: {e}")


## Push benchmarks to Genie (update evaluation rail)

Registers **BENCHMARK** curated questions on the **primary** space. If the data-rooms API errors, copy the printed Q→SQL into Genie manually.


In [ ]:
push_benchmarks(benchmarks_v1, SID_PRIMARY, 'Manufacturing v1 → primary space')


## Suggested follow-ups in the Genie UI

1. Open the space → add any failing or missing themes as **additional** curated examples (not only benchmarks).
2. Ask one benchmark question in chat and confirm the SQL matches your intent.
3. Continue to notebook **04** to explore freely, then **05**–**06** for skills and A/B testing.
